In [1]:
import pandas as pd
from pathlib import Path

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

processed_dir = project_root / "data" / "processed"

datasets = {
    "IssueDupesLite": {
        "issues": "issuedupes_issues_grouped.csv",
        "links": "issuedupes_direct_links.csv"
    },
    "Eclipse Platform": {
        "issues": "eclipse_issues_grouped.csv",
        "links": "eclipse_direct_links.csv"
    }
}

In [2]:
def prepare_training_pairs(issues, links, train_fraction=0.8):
    issues = issues.copy()
    links = links.copy()

    issues["issue_id"] = issues["issue_id"].astype(str)
    issues["text"] = issues["text"].fillna("").astype(str)
    issues["created_at"] = pd.to_datetime(
        issues["created_at"], utc=True, errors="coerce"
    )

    issues = issues[
        issues["text"].str.strip().ne("")
        & issues["created_at"].notna()
    ].sort_values(
        ["created_at", "issue_id"]
    ).reset_index(drop=True)

    split_idx = int(len(issues) * train_fraction)
    cutoff = issues.iloc[split_idx]["created_at"]

    train_issues = issues[
        issues["created_at"] < cutoff
    ].copy()

    links["source_id"] = links["source_id"].astype(str)
    links["target_id"] = links["target_id"].astype(str)

    train_ids = set(train_issues["issue_id"])

    train_links = links[
        links["source_id"].isin(train_ids)
        & links["target_id"].isin(train_ids)
        & links["source_id"].ne(links["target_id"])
    ].drop_duplicates().copy()

    text_lookup = train_issues.set_index("issue_id")["text"]

    train_links["text_a"] = train_links["source_id"].map(text_lookup)
    train_links["text_b"] = train_links["target_id"].map(text_lookup)

    return train_issues, train_links

In [3]:
training_data = {}

for name, paths in datasets.items():

    issues = pd.read_csv(
        processed_dir / paths["issues"],
        dtype={"issue_id": str},
        low_memory=False
    )

    links = pd.read_csv(
        processed_dir / paths["links"],
        dtype={"source_id": str, "target_id": str}
    )

    train_issues, train_pairs = prepare_training_pairs(
        issues, links
    )

    training_data[name] = {
        "issues": train_issues,
        "pairs": train_pairs
    }

    print(f"\n{name}")
    print("Training-period issues:", len(train_issues))
    print("Available positive pairs:", len(train_pairs))
    print("Unique source issues:", train_pairs["source_id"].nunique())
    print("Unique target issues:", train_pairs["target_id"].nunique())


IssueDupesLite
Training-period issues: 15275
Available positive pairs: 7685
Unique source issues: 7158
Unique target issues: 7685

Eclipse Platform
Training-period issues: 68124
Available positive pairs: 10959
Unique source issues: 10959
Unique target issues: 6708


In [4]:
def create_train_validation_split(train_issues, train_pairs):
    issues = train_issues.sort_values(
        ["created_at", "issue_id"]
    ).reset_index(drop=True)

    split_idx = int(len(issues) * 0.85)

    train_subset = issues.iloc[:split_idx].copy()
    validation_subset = issues.iloc[split_idx:].copy()

    train_ids = set(train_subset["issue_id"])

    fit_pairs = train_pairs[
        train_pairs["source_id"].isin(train_ids)
        & train_pairs["target_id"].isin(train_ids)
    ].copy()

    return train_subset, validation_subset, fit_pairs

In [5]:
for name, data in training_data.items():

    train_subset, validation_subset, fit_pairs = (
        create_train_validation_split(
            data["issues"],
            data["pairs"]
        )
    )

    data["train_subset"] = train_subset
    data["validation_subset"] = validation_subset
    data["fit_pairs"] = fit_pairs

    print(f"\n{name}")
    print("Training issues:", len(train_subset))
    print("Validation issues:", len(validation_subset))
    print("Training positive pairs:", len(fit_pairs))


IssueDupesLite
Training issues: 12983
Validation issues: 2292
Training positive pairs: 6439

Eclipse Platform
Training issues: 57905
Validation issues: 10219
Training positive pairs: 9477


In [6]:
for name, data in training_data.items():

    train_groups = set(
        data["train_subset"]["group_id"]
    )

    validation_groups = set(
        data["validation_subset"]["group_id"]
    )

    overlapping_groups = train_groups & validation_groups

    print(f"\n{name}")
    print(
        "Groups spanning training and validation:",
        len(overlapping_groups)
    )


IssueDupesLite
Groups spanning training and validation: 406

Eclipse Platform
Groups spanning training and validation: 423


In [7]:
def prepare_validation_benchmark(data):
    train_issues = data["train_subset"]
    val_issues = data["validation_subset"]

    issues = pd.concat(
        [train_issues, val_issues],
        ignore_index=True
    ).sort_values(
        ["created_at", "issue_id"]
    ).reset_index(drop=True)

    issue_ids = set(issues["issue_id"])

    created_lookup = issues.set_index(
        "issue_id"
    )["created_at"].to_dict()

    links = data["pairs"]

    adjacency = {}

    for source, target in links[
        ["source_id", "target_id"]
    ].itertuples(index=False, name=None):

        if (
            source not in issue_ids
            or target not in issue_ids
            or source == target
        ):
            continue

        adjacency.setdefault(source, set()).add(target)
        adjacency.setdefault(target, set()).add(source)

    relevant = {}

    for row in val_issues.itertuples(index=False):
        earlier_duplicates = {
            other
            for other in adjacency.get(row.issue_id, set())
            if created_lookup[other] < row.created_at
        }

        if earlier_duplicates:
            relevant[row.issue_id] = earlier_duplicates

    eligible = val_issues[
        val_issues["issue_id"].isin(relevant)
    ].copy()

    return {
        "all_issues": issues,
        "eligible": eligible,
        "relevant": relevant
    }

In [8]:
validation_benchmarks = {}

for name, data in training_data.items():

    benchmark = prepare_validation_benchmark(data)

    validation_benchmarks[name] = benchmark

    print(f"\n{name}")

    print(
        "Validation issues:",
        len(data["validation_subset"])
    )

    print(
        "Eligible validation queries:",
        len(benchmark["eligible"])
    )

    print(
        "Eligible percentage:",
        round(
            100 * len(benchmark["eligible"])
            / len(data["validation_subset"]),
            2
        ),
        "%"
    )


IssueDupesLite
Validation issues: 2292
Eligible validation queries: 1236
Eligible percentage: 53.93 %

Eclipse Platform
Validation issues: 10219
Eligible validation queries: 1396
Eligible percentage: 13.66 %


In [9]:
for name, data in training_data.items():

    validation_ids = set(
        data["validation_subset"]["issue_id"]
    )

    fit_pairs = data["fit_pairs"]

    leakage_count = (
        fit_pairs["source_id"].isin(validation_ids)
        | fit_pairs["target_id"].isin(validation_ids)
    ).sum()

    print(f"\n{name}")
    print(
        "Training pairs containing validation issues:",
        leakage_count
    )

    assert leakage_count == 0, (
        f"Training/validation leakage detected in {name}"
    )


IssueDupesLite
Training pairs containing validation issues: 0

Eclipse Platform
Training pairs containing validation issues: 0


In [10]:
training_data["IssueDupesLite"]["fit_pairs"].to_csv(
    processed_dir / "issuedupes_fit_pairs.csv",
    index=False
)

In [11]:
output_dir = project_root / "data" / "processed"

for name, data in training_data.items():
    filename = (
        "issuedupes_fit_pairs.csv"
        if name == "IssueDupesLite"
        else "eclipse_fit_pairs.csv"
    )

    data["fit_pairs"].to_csv(
        output_dir / filename,
        index=False
    )

    print("Saved:", filename)

Saved: issuedupes_fit_pairs.csv
Saved: eclipse_fit_pairs.csv
